In [ ]:
from firedrake import *
import numpy as np
import matplotlib.pyplot as plt
from petsc4py import PETSc

superlu False
superlu_dist True
mumps True


In [43]:
N = 16
mesh_ = RectangleMesh(N, N, np.pi, np.pi)


In [44]:
V = FunctionSpace(mesh_, "CG", 2)
W = MixedFunctionSpace([V, V, V]) #(u, v, w)


In [45]:
d_const = Constant(1.0)
rho0 = Constant(1.0)
omega = 2.0 * sqrt(1.0) 

In [46]:
dt_val = 0.01
dt = Constant(dt_val)
T_final = 1.0
n_steps = int(round(T_final/dt_val))

In [47]:
up = Function(W)
un = Function(W)

In [48]:
u_np1, v_np1, w_np1 = split(up)
u_n, v_n, w_n = split(un)

In [49]:
phi, psi, chi = TestFunctions(W)

In [50]:
u_bar = 0.5*(u_n + u_np1)
v_bar = 0.5*(v_n + v_np1)
w_bar = 0.5*(w_n + w_np1)

In [51]:
f = Constant(0.0) #problema homohenio, 3b.4 roteiro. 

In [52]:
F = (
    (u_np1 - u_n) / dt * phi * dx - v_bar * phi * dx
    + rho0 * (v_np1 - v_n) / dt * psi * dx + d_const * inner(grad(w_bar), grad(psi)) * dx
    - f * psi * dx
    + inner(grad(u_np1), grad(chi)) * dx - w_np1 * chi * dx
)


In [53]:
# CORREÇÃO 1: u = v = w = 0 no bordo (faltava bc_v)
bc_u = DirichletBC(W.sub(0), 0.0, "on_boundary")
bc_v = DirichletBC(W.sub(1), 0.0, "on_boundary")
bc_w = DirichletBC(W.sub(2), 0.0, "on_boundary")

In [54]:
x, y = SpatialCoordinate(mesh_)

In [55]:
# Condições iniciais
u0 = Function(V).interpolate(sin(x) * sin(y))          # u(0)
un.sub(0).assign(u0)
un.sub(1).assign(Constant(0.0))                        # v(0) = u_t(0) = 0

Coefficient(WithGeometry(IndexedProxyFunctionSpace(<firedrake.mesh.MeshTopology object at 0x7df9e219f260>, FiniteElement('Lagrange', triangle, 2), name=None, index=1, component=None), Mesh(VectorElement(FiniteElement('Lagrange', triangle, 1), dim=2), 51)), 120)

In [56]:
# CORREÇÃO 2: w(0) discretamente consistente com u(0):
# (w0, chi) = (grad u0, grad chi) para todo chi em V0, w0 = 0 no bordo
w0 = Function(V)
w_t, chi_t = TrialFunction(V), TestFunction(V)
solve(w_t * chi_t * dx == inner(grad(u0), grad(chi_t)) * dx, w0,
      bcs=DirichletBC(V, 0.0, "on_boundary"),
      solver_parameters={"ksp_type": "preonly", "pc_type": "lu"})
un.sub(2).assign(w0)

Coefficient(WithGeometry(IndexedProxyFunctionSpace(<firedrake.mesh.MeshTopology object at 0x7df9e219f260>, FiniteElement('Lagrange', triangle, 2), name=None, index=2, component=None), Mesh(VectorElement(FiniteElement('Lagrange', triangle, 1), dim=2), 51)), 121)

In [57]:
problem = NonlinearVariationalProblem(F, up, bcs=[bc_u, bc_v, bc_w])

In [58]:
# CORREÇÃO 3: problema linear -> um solve linear por passo, LU fatorada uma vez
solver = NonlinearVariationalSolver(
    problem,
    solver_parameters={
        "snes_type": "ksponly",           # sem iterações de Newton
        "snes_lag_jacobian": -2,          # monta o Jacobiano uma vez e não remonta
        "snes_lag_preconditioner": -2,    # fatora a LU uma vez e reaproveita
        "ksp_type": "preonly",
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "superlu_dist",  # use "petsc" se não tiver MUMPS
    },
)

In [59]:
def energia(fun):
    _, v_h, w_h = split(fun)
    return (0.5 * float(rho0) * assemble(v_h * v_h * dx)
            + 0.5 * float(d_const) * assemble(w_h * w_h * dx))

In [60]:
# CORREÇÃO 4: E0 calculada ANTES do primeiro passo
E0 = energia(un)
drifts = [0.0]

In [61]:
# CORREÇÃO 5: laço com contador inteiro (sem acúmulo de erro em t)
for n in range(n_steps):
    solver.solve()
    un.assign(up)
    drifts.append((energia(un) - E0) / E0)
 
print("Deriva máxima de energia |E_n - E_0|/E_0:", max(abs(d) for d in drifts))

Deriva máxima de energia |E_n - E_0|/E_0: 2.861575057954144e-14


In [62]:
# CORREÇÃO 6: comparação com a solução exata (usa a expressão UFL diretamente)
T = n_steps * dt_val
u_ex = cos(omega * T) * sin(x) * sin(y)
v_ex = -omega * sin(omega * T) * sin(x) * sin(y)
u_h, v_h, w_h = un.subfunctions
 
err_u = np.sqrt(assemble((u_h - u_ex) ** 2 * dx))
err_v = np.sqrt(assemble((v_h - v_ex) ** 2 * dx))
print(f"Erro L2 em T={T}: u = {err_u:.3e}, v = {err_v:.3e}")

Erro L2 em T=1.0: u = 1.074e-04, v = 1.271e-02
